In [1]:
import openai
import json


In [ ]:
# Set up your OpenAI API key
client = openai.OpenAI(api_key="xxxxxxx")

# 1. Define the tool function
def get_weather(city: str) -> str:
    return f"{city} is sunny today.  Enjoy your day!  "

# 2. tool description
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "get the weather of a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "the city to get the weather for"
                    }
                },
                "required": ["city"]
            }
        }
    }
]
# 3. function name, function mapping 
tool_map = {
    "get_weather": get_weather
}

# 4. agent loop
def run_loop(user_input: str):
    messages = [{"role": "user", 
                 "content": user_input}]
    while True:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        msg = response.choices[0].message
        if msg.tool_calls:
            messages.append(msg)

            for tool_call in msg.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                result = tool_map[name](**args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        else:
            print(msg.content)
            break
# 5. test
run_loop("What's the weather like in New York today?")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable